In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import plotly.graph_objects as go
import logging
import ast
import math
from collections import Counter
from tqdm import tqdm

from torch.nn.utils.rnn import pad_sequence

import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

# Define hyperparameters
from sklearn.utils.class_weight import compute_class_weight

import utils
import z_utils

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = "cpu"
print(device)

cuda


# 1. Data

In [2]:
spike_df = pd.read_csv('../../Data_processed/spike_tensors_x.csv')

spike_df['spike_durations'] = spike_df['spike_durations'].apply(ast.literal_eval)
spike_df['spike_durations'] = spike_df['spike_durations'].apply(lambda x: x + [0]*(4-len(x)))

spike_df['spike_type'] = spike_df['spike_type'].apply(ast.literal_eval)
spike_df['spike_type'] = spike_df['spike_type'].apply(lambda x: x + [0]*(4-len(x)))

spike_df['spike_magnitudes'] = spike_df['spike_magnitudes'].apply(ast.literal_eval)
spike_df['spike_magnitudes'] = spike_df['spike_magnitudes'].apply(lambda x: x + [0.0]*(4-len(x)))

spike_df['time'] = spike_df['spike_times_intervals'].apply(ast.literal_eval)
spike_df['time'] = spike_df['time'].apply(lambda x: x + [0]*(4-len(x)))

print(spike_df.head())

   spike_num    spike_type spike_durations              spike_magnitudes  \
0          3  [2, 2, 3, 0]    [3, 3, 4, 0]    [0.663, 0.493, 0.886, 0.0]   
1          2  [3, 2, 3, 0]    [4, 5, 0, 0]      [0.933, 1.306, 0.0, 0.0]   
2          2  [2, 2, 3, 0]    [6, 9, 0, 0]      [1.191, 1.085, 0.0, 0.0]   
3          4  [3, 2, 2, 3]    [3, 3, 3, 3]  [1.075, 0.551, 0.695, 1.164]   
4          3  [2, 3, 2, 3]    [3, 3, 5, 0]    [0.408, 0.991, 1.413, 0.0]   

                                       ID-statistics  \
0  [0.2525107493475829, 0.158, 0.2470795043314368...   
1  [0.2525107493475829, 0.158, 0.2470795043314368...   
2  [0.2525107493475829, 0.158, 0.2470795043314368...   
3  [0.2525107493475829, 0.158, 0.2470795043314368...   
4  [0.2525107493475829, 0.158, 0.2470795043314368...   

                                             weather spike_times_intervals  \
0  [0.44636145833333335, 0.3711697916666667, 0.66...          [24, 34, 38]   
1  [0.37326145833333335, 0.1712510416666667, 0.80.

In [3]:
# Get the probability of different spike num
spike_num = spike_df['spike_num'].value_counts()
spike_num = spike_num.sort_index()
total = spike_num.sum()
spike_num = spike_num / total
prob_df = pd.DataFrame(spike_num)
prob_df.columns = ['probability']
prob_df= prob_df.reset_index()
prob_df = prob_df.sort_values(by='probability', ascending=False)

print(prob_df)


    spike_num  probability
2           2     0.305778
3           3     0.269052
1           1     0.157257
4           4     0.133918
0           0     0.083342
5           5     0.040030
6           6     0.008242
7           7     0.001684
8           8     0.000406
9           9     0.000167
10         10     0.000086
11         11     0.000032
12         12     0.000007


In [4]:
# Only keep the days with spike_num < 5
spike_df = spike_df[spike_df['spike_num'] < 5]
spike_num = spike_df['spike_num'].value_counts()
spike_num = spike_num.sort_index()
total = spike_num.sum()
spike_num = spike_num / total
prob_df = pd.DataFrame(spike_num)
prob_df.columns = ['probability']
prob_df= prob_df.reset_index()
prob_df = prob_df.sort_values(by='probability', ascending=False)

print(prob_df)


   spike_num  probability
2          2     0.322093
3          3     0.283407
1          1     0.165647
4          4     0.141064
0          0     0.087789


In [5]:
all_durations = [duration for sublist in spike_df['spike_durations'] for duration in sublist if duration != 0]

# Count the occurrences of each duration
duration_counts = Counter(all_durations)

# Calculate the total number of non-zero durations
total_durations = sum(duration_counts.values())

# Calculate the probability of each duration
duration_probabilities = {duration: count / total_durations for duration, count in duration_counts.items()}

# Convert to a DataFrame for better readability
probability_df = pd.DataFrame(list(duration_probabilities.items()), columns=['Duration', 'Probability'])

# Sort by Duration
probability_df = probability_df.sort_values(by='Probability', ascending=False).reset_index(drop=True)

print(probability_df)

# Add together the probabilities for 10 most common durations
top_10_probability = probability_df['Probability'].iloc[:13].sum()
print(f'The 13 most common durations account for {top_10_probability:.2%} of all non-zero durations')


    Duration   Probability
0          3  4.396302e-01
1          5  1.011166e-01
2          4  8.101210e-02
3          6  7.845331e-02
4          7  5.036503e-02
5          8  4.227612e-02
6          9  3.424870e-02
7         10  2.768852e-02
8         11  2.366418e-02
9         12  1.988363e-02
10        13  1.642147e-02
11        14  1.374648e-02
12        15  1.110663e-02
13         2  8.598698e-03
14        16  8.511165e-03
15         1  8.457382e-03
16        17  6.150865e-03
17        18  4.768835e-03
18        19  3.672365e-03
19        20  2.994449e-03
20        21  2.450606e-03
21        22  2.064415e-03
22        23  1.746647e-03
23        24  1.510093e-03
24        25  1.252272e-03
25        26  1.116042e-03
26        27  9.972252e-04
27        28  9.186307e-04
28        29  8.297111e-04
29        30  7.367846e-04
30        31  6.717516e-04
31        32  5.782087e-04
32        33  4.507624e-04
33        34  3.555244e-04
34        48  3.054397e-04
35        35  2.535057e-04
3

In [6]:
top_durations = set(probability_df.nlargest(7, 'Probability')['Duration'])

# Function to filter and reorder durations in each row
def filter_and_reorder_with_indices(duration_list, type_list, magnitude_list, time_list):
    # Ensure the lists are of equal length
    min_length = 4
    
    # Trim the lists if they have inconsistent lengths
    duration_list = duration_list[:min_length]
    type_list = type_list[:min_length]
    magnitude_list = magnitude_list[:min_length]
    time_list = time_list[:min_length]
    
    # Indices of the durations to keep based on the top probabilities
    keep_indices = [i for i, d in enumerate(duration_list) if d in top_durations]

    # Filter and keep only the relevant indices, moving zeros to the end
    filtered_durations = [duration_list[i]-2 for i in keep_indices] + [0] * (min_length - len(keep_indices))
    filtered_types = [type_list[i] for i in keep_indices] + [0] * (min_length - len(keep_indices))
    filtered_magnitudes = [magnitude_list[i] for i in keep_indices] + [0.0] * (min_length - len(keep_indices))
    filtered_time = [time_list[i] for i in keep_indices] + [0] * (min_length - len(keep_indices))

    return filtered_durations, filtered_types, filtered_magnitudes, filtered_time

spike_df[['filtered_spike_durations', 'filtered_spike_types', 'filtered_spike_magnitudes', 'filtered_time']] = spike_df.apply(
    lambda row: filter_and_reorder_with_indices(row['spike_durations'], row['spike_type'], row['spike_magnitudes'], row['time']),
    axis=1, result_type='expand'
)

print(spike_df.head())

   spike_num    spike_type spike_durations              spike_magnitudes  \
0          3  [2, 2, 3, 0]    [3, 3, 4, 0]    [0.663, 0.493, 0.886, 0.0]   
1          2  [3, 2, 3, 0]    [4, 5, 0, 0]      [0.933, 1.306, 0.0, 0.0]   
2          2  [2, 2, 3, 0]    [6, 9, 0, 0]      [1.191, 1.085, 0.0, 0.0]   
3          4  [3, 2, 2, 3]    [3, 3, 3, 3]  [1.075, 0.551, 0.695, 1.164]   
4          3  [2, 3, 2, 3]    [3, 3, 5, 0]    [0.408, 0.991, 1.413, 0.0]   

                                       ID-statistics  \
0  [0.2525107493475829, 0.158, 0.2470795043314368...   
1  [0.2525107493475829, 0.158, 0.2470795043314368...   
2  [0.2525107493475829, 0.158, 0.2470795043314368...   
3  [0.2525107493475829, 0.158, 0.2470795043314368...   
4  [0.2525107493475829, 0.158, 0.2470795043314368...   

                                             weather spike_times_intervals  \
0  [0.44636145833333335, 0.3711697916666667, 0.66...          [24, 34, 38]   
1  [0.37326145833333335, 0.1712510416666667, 0.80.

In [7]:
all_durations = [duration for sublist in spike_df['filtered_spike_durations'] for duration in sublist if duration != 0]
# Count the occurrences of each duration
duration_counts = Counter(all_durations)

# Calculate the total number of non-zero durations
total_durations = sum(duration_counts.values())

# Calculate the probability of each duration
duration_probabilities = {duration: count / total_durations for duration, count in duration_counts.items()}

# Convert to a DataFrame for better readability
probability_df = pd.DataFrame(list(duration_probabilities.items()), columns=['Duration', 'Probability'])

# Sort by Duration
probability_df = probability_df.sort_values(by='Probability', ascending=False).reset_index(drop=True)

print(probability_df)

   Duration  Probability
0         1     0.531531
1         3     0.122254
2         2     0.097947
3         4     0.094853
4         5     0.060893
5         6     0.051114
6         7     0.041408


In [8]:
def count_non_zero_durations(duration_list):
    return sum(1 for d in duration_list if d != 0)

# Update the spike_num column to reflect the count of non-zero durations
spike_df['spike_num'] = spike_df['filtered_spike_durations'].apply(count_non_zero_durations)

# Display the first few rows to verify the changes
spike_df.head(20)

,spike_num,spike_type,spike_durations,spike_magnitudes,ID-statistics,weather,spike_times_intervals,date_sin,date_cos,time,filtered_spike_durations,filtered_spike_types,filtered_spike_magnitudes,filtered_time
0,3,"[2, 2, 3, 0]","[3, 3, 4, 0]","[0.663, 0.493, 0.886, 0.0]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.44636145833333335, 0.3711697916666667, 0.66...","[24, 34, 38]",-0.977848,0.209315,"[24, 34, 38, 0]","[1, 1, 2, 0]","[2, 2, 3, 0]","[0.663, 0.493, 0.886, 0.0]","[24, 34, 38, 0]"
1,2,"[3, 2, 3, 0]","[4, 5, 0, 0]","[0.933, 1.306, 0.0, 0.0]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.37326145833333335, 0.1712510416666667, 0.80...","[20, 36]",-0.974100,0.226116,"[20, 36, 0, 0]","[2, 3, 0, 0]","[3, 2, 0, 0]","[0.933, 1.306, 0.0, 0.0]","[20, 36, 0, 0]"
2,2,"[2, 2, 3, 0]","[6, 9, 0, 0]","[1.191, 1.085, 0.0, 0.0]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.35165416666666666, 0.17661875000000002, 0.7...","[20, 37]",-0.970064,0.242850,"[20, 37, 0, 0]","[4, 7, 0, 0]","[2, 2, 0, 0]","[1.191, 1.085, 0.0, 0.0]","[20, 37, 0, 0]"
3,4,"[3, 2, 2, 3]","[3, 3, 3, 3]","[1.075, 0.551, 0.695, 1.164]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.392646875, 0.22844895833333334, 0.79748125]","[17, 23, 34, 38]",-0.965740,0.259512,"[17, 23, 34, 38]","[1, 1, 1, 1]","[3, 2, 2, 3]","[1.075, 0.551, 0.695, 1.164]","[17, 23, 34, 38]"
4,3,"[2, 3, 2, 3]","[3, 3, 5, 0]","[0.408, 0.991, 1.413, 0.0]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.45284687500000004, 0.3914020833333333, 0.65...","[18, 25, 36]",-0.961130,0.276097,"[18, 25, 36, 0]","[1, 1, 3, 0]","[2, 3, 2, 0]","[0.408, 0.991, 1.413, 0.0]","[18, 25, 36, 0]"
5,2,"[2, 2, 0, 0]","[8, 3, 0, 0]","[0.784, 0.377, 0.0, 0.0]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.49727499999999997, 0.374875, 0.759745833333...","[19, 43]",-0.956235,0.292600,"[19, 43, 0, 0]","[6, 1, 0, 0]","[2, 2, 0, 0]","[0.784, 0.377, 0.0, 0.0]","[19, 43, 0, 0]"
6,1,"[3, 2, 0, 0]","[7, 0, 0, 0]","[1.6720000000000002, 0.0, 0.0, 0.0]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.51195625, 0.274325, 0.8019531249999999]",[38],-0.951057,0.309017,"[38, 0, 0, 0]","[5, 0, 0, 0]","[3, 0, 0, 0]","[1.6720000000000002, 0.0, 0.0, 0.0]","[38, 0, 0, 0]"
7,2,"[2, 2, 0, 0]","[3, 3, 0, 0]","[0.406, 0.822, 0.0, 0.0]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.47235, 0.09946458333333334, 0.9436062500000...","[34, 40]",-0.945596,0.325342,"[34, 40, 0, 0]","[1, 1, 0, 0]","[2, 2, 0, 0]","[0.406, 0.822, 0.0, 0.0]","[34, 40, 0, 0]"
8,1,"[2, 3, 2, 0]","[6, 16, 0, 0]","[0.798, 1.879, 0.0, 0.0]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.45446458333333334, 0.09321770833333333, 0.8...","[22, 33]",-0.939856,0.341571,"[22, 33, 0, 0]","[4, 0, 0, 0]","[2, 0, 0, 0]","[0.798, 0.0, 0.0, 0.0]","[22, 0, 0, 0]"
9,2,"[2, 2, 2, 3, 2]","[7, 3, 17, 0]","[1.189, 0.657, 2.6060000000000003, 0.0]","[0.2525107493475829, 0.158, 0.2470795043314368...","[0.4468854166666667, 0.2341895833333333, 0.917...","[20, 28, 32]",-0.933837,0.357698,"[20, 28, 32, 0]","[5, 1, 0, 0]","[2, 2, 0, 0]","[1.189, 0.657, 0.0, 0.0]","[20, 28, 0, 0]"


In [9]:
# See the max value for spike_magnitude
max_magnitude = max(spike_df['spike_magnitudes'].apply(max))
print(max_magnitude)

61.311


In [10]:
all_values = [value for tensor in spike_df['filtered_time'] for value in tensor]

# Convert to a set to get the unique values and count them
unique_values = set(all_values)

# Number of unique values
num_unique_values = len(unique_values)
print(unique_values)

# Display the result
print(f"The number of different individual values in 'filtered_time' column is: {num_unique_values}")

{0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46}
The number of different individual values in 'filtered_time' column is: 47


In [11]:
all_values = [value for tensor in spike_df['filtered_spike_durations'] for value in tensor]

# Convert to a set to get the unique values and count them
unique_values = set(all_values)

# Number of unique values
num_unique_values = len(unique_values)

# Display the result
print(f"The number of different individual values in 'spike_durations' column is: {num_unique_values}")

The number of different individual values in 'spike_durations' column is: 8


In [12]:
print(spike_df.head())

   spike_num    spike_type spike_durations              spike_magnitudes  \
0          3  [2, 2, 3, 0]    [3, 3, 4, 0]    [0.663, 0.493, 0.886, 0.0]   
1          2  [3, 2, 3, 0]    [4, 5, 0, 0]      [0.933, 1.306, 0.0, 0.0]   
2          2  [2, 2, 3, 0]    [6, 9, 0, 0]      [1.191, 1.085, 0.0, 0.0]   
3          4  [3, 2, 2, 3]    [3, 3, 3, 3]  [1.075, 0.551, 0.695, 1.164]   
4          3  [2, 3, 2, 3]    [3, 3, 5, 0]    [0.408, 0.991, 1.413, 0.0]   

                                       ID-statistics  \
0  [0.2525107493475829, 0.158, 0.2470795043314368...   
1  [0.2525107493475829, 0.158, 0.2470795043314368...   
2  [0.2525107493475829, 0.158, 0.2470795043314368...   
3  [0.2525107493475829, 0.158, 0.2470795043314368...   
4  [0.2525107493475829, 0.158, 0.2470795043314368...   

                                             weather spike_times_intervals  \
0  [0.44636145833333335, 0.3711697916666667, 0.66...          [24, 34, 38]   
1  [0.37326145833333335, 0.1712510416666667, 0.80.

In [13]:
ids = spike_df['ID-statistics'].unique()
train_ids = np.random.choice(ids, size=3000, replace=False)
val_ids = np.random.choice([id for id in ids if id not in train_ids], size=1000, replace=False)
test_ids = [id for id in ids if id not in train_ids and id not in val_ids]

train_df = spike_df[spike_df['ID-statistics'].isin(train_ids)]
val_df = spike_df[spike_df['ID-statistics'].isin(val_ids)]
test_df = spike_df[spike_df['ID-statistics'].isin(test_ids)]

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(1806439, 14)
(605246, 14)
(505628, 14)


In [14]:
sequence_length = 14

# spike_int_columns = ['spike_num', 'filtered_spike_types', 'filtered_spike_durations',  'filtered_time']
# spike_float_columns = ['filtered_spike_magnitudes']
num_column = 'spike_num'
type_column = 'filtered_spike_types'
duration_column = 'filtered_spike_durations'
time_column = 'filtered_time'
magnitude_column = 'filtered_spike_magnitudes'
weather_column = 'weather'
date_column = ['date_sin', 'date_cos']
id_column = 'ID-statistics'

def prepare_sequences(df, sequence_length):

    spike_num_sequences = []
    spike_type_sequences = []
    spike_duration_sequences = []
    spike_time_sequences = []
    spike_magnitude_sequences = []

    weather_sequences = []
    date_sequences = []
    id_sequences = []

    ids = df['ID-statistics'].unique()

    for id in tqdm(ids, desc = "Processing LCLids"): 
        df_group = df[df['ID-statistics'] == id]

        spike_num_data = df_group[num_column].values
        spike_type_data = df_group[type_column].values
        spike_duration_data = df_group[duration_column].values
        spike_time_data = df_group[time_column].values
        spike_magnitude_data = df_group[magnitude_column].values

        weather_data = df_group[weather_column].values
        date_data = df_group[date_column].values
        id_data = df_group[id_column].values

        i=0
        
        while i+sequence_length < len(spike_num_data):
            spike_num_sequences.append(spike_num_data[i:i+sequence_length])
            spike_type_sequences.append(spike_type_data[i:i+sequence_length])
            spike_duration_sequences.append(spike_duration_data[i:i+sequence_length])
            spike_time_sequences.append(spike_time_data[i:i+sequence_length])
            spike_magnitude_sequences.append(spike_magnitude_data[i:i+sequence_length])

            weather_sequences.append(weather_data[i:i+sequence_length])
            date_sequences.append(date_data[i:i+sequence_length])
            id_sequences.append(id_data[i:i+sequence_length])

            n = np.random.randint(1, sequence_length)
            i += n
    
    return np.array(spike_num_sequences), np.array(spike_type_sequences), np.array(spike_duration_sequences), np.array(spike_time_sequences), np.array(spike_magnitude_sequences), np.array(weather_sequences), np.array(date_sequences), np.array(id_sequences)



In [15]:
train_spike_num_sequences, train_spike_type_sequences, train_spike_duration_sequences, train_spike_time_sequences, train_spike_magnitude_sequences, train_weather_sequences, train_date_sequences, train_id_sequences = prepare_sequences(train_df, sequence_length)
val_spike_num_sequences, val_spike_type_sequences, val_spike_duration_sequences, val_spike_time_sequences, val_spike_magnitude_sequences, val_weather_sequences, val_date_sequences, val_id_sequences = prepare_sequences(val_df, sequence_length)

torch.save(train_spike_num_sequences, '../../Data_processed/VAE/sgen_train_spike_num.pt')
torch.save(train_spike_type_sequences, '../../Data_processed/VAE/sgen_train_spike_type.pt')
torch.save(train_spike_duration_sequences, '../../Data_processed/VAE/sgen_train_spike_duration.pt')
torch.save(train_spike_time_sequences, '../../Data_processed/VAE/sgen_train_spike_time.pt')
torch.save(train_spike_magnitude_sequences, '../../Data_processed/VAE/sgen_train_spike_magnitude.pt')
torch.save(train_weather_sequences, '../../Data_processed/VAE/sgen_train_weather.pt')
torch.save(train_date_sequences, '../../Data_processed/VAE/sgen_train_date.pt')
torch.save(train_id_sequences, '../../Data_processed/VAE/sgen_train_id.pt')

torch.save(val_spike_num_sequences, '../../Data_processed/VAE/sgen_val_spike_num.pt')
torch.save(val_spike_type_sequences, '../../Data_processed/VAE/sgen_val_spike_type.pt')
torch.save(val_spike_duration_sequences, '../../Data_processed/VAE/sgen_val_spike_duration.pt')
torch.save(val_spike_time_sequences, '../../Data_processed/VAE/sgen_val_spike_time.pt')
torch.save(val_spike_magnitude_sequences, '../../Data_processed/VAE/sgen_val_spike_magnitude.pt')
torch.save(val_weather_sequences, '../../Data_processed/VAE/sgen_val_weather.pt')
torch.save(val_date_sequences, '../../Data_processed/VAE/sgen_val_date.pt')
torch.save(val_id_sequences, '../../Data_processed/VAE/sgen_val_id.pt')

Processing LCLids: 100%|██████████| 1000/1000 [00:17<00:00, 56.64it/s]


In [16]:
# train_spike_int_sequences, train_spike_float_sequences, train_weather_sequences, train_date_sequences, train_id_sequences = prepare_sequences(train_df, sequence_length) 
# val_spike_int_sequences, val_spike_float_sequences, val_weather_sequences, val_date_sequences, val_id_sequences = prepare_sequences(val_df, sequence_length)

# torch.save(train_spike_int_sequences, '../../Data_processed/VAE/sgen_train_int_spike.pt')
# torch.save(train_spike_float_sequences, '../../Data_processed/VAE/sgen_train_float_spike.pt')
# torch.save(train_weather_sequences, '../../Data_processed/VAE/sgen_train_weather.pt')
# torch.save(train_date_sequences, '../../Data_processed/VAE/sgen_train_date.pt')
# torch.save(train_id_sequences, '../../Data_processed/VAE/sgen_train_id.pt')

# torch.save(val_spike_int_sequences, '../../Data_processed/VAE/sgen_val_int_spike.pt')
# torch.save(val_spike_float_sequences, '../../Data_processed/VAE/sgen_val_float_spike.pt')
# torch.save(val_weather_sequences, '../../Data_processed/VAE/sgen_val_weather.pt')
# torch.save(val_date_sequences, '../../Data_processed/VAE/sgen_val_date.pt')
# torch.save(val_id_sequences, '../../Data_processed/VAE/sgen_val_id.pt')

# test_df.to_csv('../../Data_processed/VAE/sgen_test.csv', index=False)